# LLMs examples and prompting
This notebook connects to the APIs for different LLMs and asks the relevant questions.

Make sure a GraphDB instance is running.

## Set up libraries, APIs, etc.

In [1]:
import os
import re
import pandas as pd
from openai import OpenAI
from functions.sparql_requests import sparql_select, sparql_update

Set up the API client using a locally stored API key.

In [2]:
client = OpenAI(api_key = os.environ['OPENAI1'])

## First LLM call: raw
First, the LLM is prompted for neighboring NUTS-regions without any RAG-style expansions. Different prompting techniques are tested.\
The `write_sysmsg` function writes a prompt for the system message of the LLM and expandes it step by step for more detailes. Then, different combinations of the system message building blocks are assessed.

### x.x Define system message

In [3]:
def write_sysmsg(kwds=[]):
    assert not (("only_names" in kwds) & ("only_codes" in kwds)), "codes and names cannot be both a requirement at the same time!"
    
    sysmsg = ""
    if "intro" in kwds:
        sysmsg += "You are a helpful assistant."
        
    if "details" in kwds:
        sysmsg = sysmsg[:-1]
        sysmsg += " that specializes in the European NUTS regions."
        
    if "task_specific" in kwds:
        sysmsg += "\nYour task is to identify neighbors of the region the user is asking about."

    if "consise" in kwds:
        sysmsg += "\nYour answer should be consise."

    if "only_names" in kwds:
        sysmsg += "\nYour answer should only contain the relevant regions' names. Nothing else."

    if "only_codes" in kwds:
        sysmsg += "\nYour answer should only contain the relevant regions' NUTS-codes. Nothing else."

    if "python_list" in kwds:
        sysmsg += "\nFormat your answer as a python list."

    return sysmsg

### x.x Set up 

In [4]:
# does not make sense to mix only_names/codes with consise
# 11 permutations
permutations = [
    ["intro", "consise"],
    # ["intro", "consise", "only_names"],
    ["intro", "consise", "only_codes"],
    
    ["intro", "details", "consise"],
    ["intro", "details", "task_specific", "consise"],

    # ["intro", "details", "only_names"],
    ["intro", "details", "only_codes"],

    # ["intro", "details", "task_specific", "only_names"],
    ["intro", "details", "task_specific", "only_codes"],
        
    # ["intro", "details", "task_specific", "only_names", "python_list"],
    ["intro", "details", "task_specific", "only_codes", "python_list"]
]

### x.x Set up test codes
Different NUTS-codes are chosen for all levels and all over Europe. Also a function is defined to retrieve the name of a NUTS-region.

In [5]:
# 18 codes
codes = ["CZ", "BG",
         "FR1", "DEG", "PL7", "ITI",
         "PT18", "FRK2", "CZ03", "DEF0", "RO41",
         "DEB3I", "PL515", "LT022", "SE331", "IE051", "ES615", "FRK21"
        ]


def construct_code_with_name(code):
    query = f'''
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    SELECT ?label WHERE {{
        ?code skos:notation "{code}" .
        ?code skos:altLabel ?label .
    }}'''
    name = sparql_select(query)['label'].values.tolist()[-1]
    return name

### x.x Set up a DataFrame for requests

In [6]:
def set_up_requests(permutations=permutations, codes=codes):
    llist = []
    
    for permutation in permutations:
        sysmsg = write_sysmsg(permutation)
        sysmsg_label = ','.join(permutation)

        for code in codes:
            usermsg = f"Give me all nuts regions that share a border with the region {code} and have the same NUTS level"

            code_with_name = construct_code_with_name(code)

            row = [sysmsg, sysmsg_label, code, code+", "+code_with_name, usermsg]

            llist.append(row)

    df = pd.DataFrame(llist, columns=["sysmsg", "sysmsg_permutation", "code", "code_with_name", "usermsg"])
    return df

df_prompts = set_up_requests()
df_prompts.head(2)

,sysmsg,sysmsg_permutation,code,code_with_name,usermsg
0,You are a helpful assistant.\nYour answer shou...,"intro,consise",CZ,"CZ, ČESKÁ REPUBLIKA",Give me all nuts regions that share a border w...
1,You are a helpful assistant.\nYour answer shou...,"intro,consise",BG,"BG, Bulgaria",Give me all nuts regions that share a border w...


### x.x Run the requests through the OpenAI API

In [7]:
def run_client(sysmsg, usermsg, temperature=0, model="gpt-3.5-turbo"):
    completion = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=300,
        n=1,
        messages=[
            {"role": "system", "content": sysmsg},
            {"role": "user", "content": usermsg}
            ]
        )

    return completion.choices[0].message.content, completion.usage.prompt_tokens, completion.usage.completion_tokens

In [8]:
# # run for all prompts

# temp = df_prompts.apply(lambda x: run_client(x['sysmsg'], x['usermsg']), axis=1)
# df_prompts[["answer", "tokens_in", "tokens_out"]] = pd.DataFrame(temp.values.tolist(), index=temp.index)

In [9]:
# df_prompts.to_csv("data/raw_prompting_answers.csv", index=False)

In [10]:
# read saved file
df_prompts = pd.read_csv("data/raw_prompting_answers.csv")

### x.x Get codes out of answer
The answer contains NUTS-codes. These are normally capitalized, which is important since natural words lit 'at' could be mistaken as codes otherwise.\
A variable `total_codes` is defined which contains all NUTS codes. These are matched in the LLMs answer. However the LLM also makes up NUTS-codes that do not exists. In order to evaluate the LLMs performance we also want to detect made up codes. This is done via a regular expression where the word NUTS is filtered and then the expressions with two capital letters and up to 3 capital letters or numbers are detected.

In [11]:
query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT ?codename WHERE {
    ?code rdf:type skos:Concept .
    ?code skos:notation ?codename.
}
"""
total_codes = sparql_select(query).codename.tolist()

In [12]:
def get_answer_codes(org_code, answer):
    # remove the original code from the answer
    answer = answer.replace(" "+org_code+" ", " __ ")
    codes_in_answer = []

    pattern = rf'(?!NUTS)(?!UTS)(?!TS)[A-Z]{{2}}[A-Z0-9]{{0,3}}'
    matches = re.findall(pattern, answer)

    # check for each existing code if
    for code in total_codes:
        if code in matches:
            codes_in_answer.append(code)

    matches = set(matches)
    codes_in_answer = set(codes_in_answer)

    wrong_codes = matches - codes_in_answer

    return list(codes_in_answer), list(wrong_codes)

In [13]:
with open("sparql/get_all_neighborhoods_same_level.sparql", "r") as f:
    query_allNeighborhoodsSameLevel = f.read()
# print(query_allNeighborhoodsSameLevel)

In [14]:
def get_correct_codes(org_code):
    query = query_allNeighborhoodsSameLevel.replace("PLACEHOLDER", org_code)
    # print(query)
    df = sparql_select(query)
    return df['name'].values.tolist()

In [15]:
df_prompts['correct_codes'] = df_prompts['code'].apply(get_correct_codes)

In [16]:
temp = df_prompts.apply(lambda x: get_answer_codes(x['code'], x['answer']), axis=1)
df_prompts[['hits', 'wrong_hits']] = pd.DataFrame(temp.values.tolist())

## Parsing
Now the raw LLM approach will be compared to the simplyfied RAG approach. We parse the relevant code from the question, create our query to retrieve the correct answers, and then give the answers to the LLM. We can then see if it is parsed correctly.

For parsing the question for the relevant codes, the `get_answer_codes` function from above could be used. I a more realistic setting, an LLM call could be used to retrieve the NUTS coude from the question. Since `get_answer_codes` should yield the code that was inserted from the `code` column, we can directly work with this one.

In [17]:
df_prompts.head(2)

,sysmsg,sysmsg_permutation,code,code_with_name,usermsg,answer,tokens_in,tokens_out,correct_codes,hits,wrong_hits
0,You are a helpful assistant.\nYour answer shou...,"intro,consise",CZ,"CZ, ČESKÁ REPUBLIKA",Give me all nuts regions that share a border w...,The regions that share a border with CZ at the...,45,25,"[DE, PL, AT, SK]","[SK, DE, AT, PL]",[]
1,You are a helpful assistant.\nYour answer shou...,"intro,consise",BG,"BG, Bulgaria",Give me all nuts regions that share a border w...,The regions that share a border with region BG...,45,30,"[RO, EL]","[TR, RS, RO, MK]",[GR]


## Third LLM call: the answer

In [18]:
sysmsg_rag = "You are a helpful assistant that answers a users question based only on the provided context. If you do not know the answer just say that you dont know it. Do not make anything up. Your answer should only contain the relevant information."

def format_usermsg_rag(row):
    usermsg_rag = f"""The user asked the following question:

Give me all NUTS regions that share a border with the region PLACEHOLDER_CODE.
    
This is the answer retrieved from a database:
PLACEHOLDER_ANSWER
    """
    
    usermsg = usermsg_rag.replace("PLACEHOLDER_CODE", row['code'])
    usermsg = usermsg.replace("PLACEHOLDER_ANSWER", str(row['correct_codes']))
    return usermsg
    # print(usermsg)

df_prompts['usermsg_rag'] = df_prompts.apply(format_usermsg_rag, axis=1)                                         

In [19]:
# Run all requests

# temp = df_prompts['usermsg_rag'].apply(lambda x: run_client(sysmsg_rag, x))
# df_prompts[["answer_rag", "tokens_in_rag", "tokens_out_rag"]] = pd.DataFrame(temp.values.tolist(), index=temp.index)

# temp = df_prompts.apply(lambda x: get_answer_codes(x['code'], x['answer_rag']), axis=1)
# df_prompts[['hits_rag', 'wrong_hits_rag']] = pd.DataFrame(temp.values.tolist())

# df_prompts.to_csv("data/126_samples_all.csv", index=False)

In [20]:
df_prompts = pd.read_csv("data/126_samples_all.csv")

columns = ['correct_codes', 'hits', 'wrong_hits', 'hits_rag', 'wrong_hits_rag']

for col in columns:
    df_prompts[col] = df_prompts[col].apply(eval)


In [21]:
df_prompts.head(1)

,sysmsg,sysmsg_permutation,code,code_with_name,usermsg,answer,tokens_in,tokens_out,correct_codes,hits,wrong_hits,usermsg_rag,answer_rag,tokens_in_rag,tokens_out_rag,hits_rag,wrong_hits_rag
0,You are a helpful assistant.\nYour answer shou...,"intro,consise",CZ,"CZ, ČESKÁ REPUBLIKA",Give me all nuts regions that share a border w...,The regions that share a border with CZ at the...,45,25,"[DE, PL, AT, SK]","[SK, PL, AT, DE]",[],The user asked the following question:\n\nGive...,The NUTS regions that share a border with the ...,107,23,"[SK, PL, AT, DE]",[]


# Results

In [22]:
def get_hit_ratio(hitlist, correct_codes, wrong_hits):
    hits = set(hitlist)
    correct = set(correct_codes)
    wrong = set(wrong_hits)

    correct_hits = hits.intersection(correct)
    
    print(f"{len(correct_hits) / len(correct) *100} %")

    print(f"Additionaly, {len(wrong)} nonexistent codes were given")


_ = df_prompts.apply(lambda x: get_hit_ratio(x['hits'], x['correct_codes'], x['wrong_hits']), axis=1)

100.0 %
Additionaly, 0 nonexistent codes were given
50.0 %
Additionaly, 1 nonexistent codes were given
60.0 %
Additionaly, 0 nonexistent codes were given
20.0 %
Additionaly, 2 nonexistent codes were given
33.33333333333333 %
Additionaly, 0 nonexistent codes were given
33.33333333333333 %
Additionaly, 1 nonexistent codes were given
40.0 %
Additionaly, 0 nonexistent codes were given
14.285714285714285 %
Additionaly, 1 nonexistent codes were given
28.57142857142857 %
Additionaly, 1 nonexistent codes were given
0.0 %
Additionaly, 3 nonexistent codes were given
25.0 %
Additionaly, 0 nonexistent codes were given
8.333333333333332 %
Additionaly, 0 nonexistent codes were given
14.285714285714285 %
Additionaly, 0 nonexistent codes were given
33.33333333333333 %
Additionaly, 3 nonexistent codes were given
33.33333333333333 %
Additionaly, 2 nonexistent codes were given
50.0 %
Additionaly, 1 nonexistent codes were given
20.0 %
Additionaly, 0 nonexistent codes were given
16.666666666666664 %
Additi

In [24]:
_ = df_prompts.apply(lambda x: get_hit_ratio(x['hits_rag'], x['correct_codes'], x['wrong_hits_rag']), axis=1)

100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
0.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additionaly, 0 nonexistent codes were given
100.0 %
Additi